|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>The async engine<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: build the engine loop<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import asyncio
import time

STEP_S = 0.010
print('ready')

Build the engine loop.

One coroutine steps the model forever. Requests arrive on a queue, and tokens
leave on one queue for each request. Nothing blocks in either direction.

This is stage 15, with `asyncio.sleep` in the place of the model. Make the
concurrency correct before you add the GPU.

# Exercise 1: submit, step, stream

Three pieces: a way in, a loop, and a way out. The loop is the only thing
that touches the model.

In [ ]:
from dataclasses import dataclass

@dataclass(eq=False)
class Running:
  tokens_left: int
  stream: asyncio.Queue       # the tokens for the client. None ends the stream.

class Engine:
  """One loop steps the model. Each client reads its own stream."""
  def __init__(self, max_running=8):
    self.waiting = asyncio.Queue()
    self.running = {}            # request id -> Running
    self.max_running = max_running
    self.stopped = False

  async def submit(self, request_id, num_tokens):
    """The HTTP handler calls this. It must return at once."""
    stream = asyncio.Queue()
    
    return stream

  def _admit(self):
    """Move waiting requests to running while there is space."""
    

  def _emit_tokens(self):
    """Give each running request its token. Close the streams that finish."""
    

  async def loop(self):
    """The only code that touches the model. One step serves all requests."""
    while not self.stopped:
      self._admit()
      await asyncio.sleep(STEP_S)              # the step
      self._emit_tokens()

  def cancel(self, request_id):
    """A client hung up. Free the slot NOW, not when the request would end."""
    

async def stop_engine(engine, task):
  engine.stopped = True
  await asyncio.sleep(0.05)
  task.cancel()

async def read_stream(stream, start):
  """Read tokens until None.
  -> (time to the first token, time to the end), from start."""
  first = None
  while await stream.get() is not None:
    if first is None:
      first = time.perf_counter() - start
  return first, time.perf_counter() - start

async def client(request_id, num_tokens):
  start = time.perf_counter()
  stream = await engine.submit(request_id, num_tokens)
  return await read_stream(stream, start)

engine = Engine(max_running=4)
task = asyncio.create_task(engine.loop())
latencies = await asyncio.gather(*[client(request_id, 10) for request_id in range(8)])
await stop_engine(engine, task)
ttfts = [first for first, _ in latencies]
print(f'{len(latencies)} clients done')
print(f'TTFT  min {min(ttfts):.3f}s  max {max(ttfts):.3f}s')

# Exercise 2: somebody closes the tab

Watch the number of occupied slots against time. Two clients hang up after
five tokens. Their slots must come back at once, and not at the time when the
request would have finished.

In [ ]:
engine = Engine(max_running=4)
task = asyncio.create_task(engine.loop())
slots_in_use = []

async def watch_slots():
  for _ in range(40):
    slots_in_use.append(len(engine.running))
    await asyncio.sleep(STEP_S)

async def quit_early(request_id, num_tokens, quit_after):
  stream = await engine.submit(request_id, num_tokens)
  for _ in range(quit_after):
    if await stream.get() is None:
      return
  # The tab closed. What must happen here?
  

async def read_to_end(request_id, num_tokens):
  stream = await engine.submit(request_id, num_tokens)
  await read_stream(stream, time.perf_counter())

await asyncio.gather(watch_slots(),
                     *[quit_early(request_id, 30, 5) for request_id in (0, 1)],
                     *[read_to_end(request_id, 30) for request_id in (2, 3)])
await stop_engine(engine, task)
print('slots in use over time:', slots_in_use[:20])
print(f'\npeak {max(slots_in_use)}')

# Exercise 3: against the obvious design

Now the version where the model runs inside the event loop. Measure time to
first token, not throughput.

In [ ]:
async def blocking_ttfts(num_clients, tokens):
  """The model runs inside the event loop: time.sleep, not asyncio.sleep."""
  start = time.perf_counter()
  ttfts = []
  async def client():
    first = None
    for _ in range(tokens):
      
    ttfts.append(first)
  await asyncio.gather(*[client() for _ in range(num_clients)])
  return ttfts

async def engine_ttfts(num_clients, tokens):
  engine = Engine(max_running=num_clients)
  task = asyncio.create_task(engine.loop())
  start = time.perf_counter()
  async def client(request_id):
    
  ttfts = await asyncio.gather(*[client(request_id) for request_id in range(num_clients)])
  await stop_engine(engine, task)
  return list(ttfts)

async def measure(design, num_clients=8, tokens=15):
  start = time.perf_counter()
  run = blocking_ttfts if design == 'blocking' else engine_ttfts
  ttfts = sorted(await run(num_clients, tokens))
  return ttfts, time.perf_counter() - start

for design in ('blocking', 'engine'):
  ttfts, wall = await measure(design)
  print(f'{design:>9}: wall {wall:5.2f}s  TTFT p50 {ttfts[len(ttfts)//2]:.3f}s  '
        f'p99 {ttfts[-1]:.3f}s')

### Before you open the solution

1. `submit` returns a queue rather than the tokens. Why can it not just
   await the result and return it?
2. Delete the body of `cancel` and rerun Exercise 2. What does the
   occupancy trace look like, and what is the server holding?
3. The blocking design in Exercise 3 uses `asyncio.gather`, which looks
   concurrent. Why is it not?
4. The engine sends `None` to end a stream. What would break if it simply
   stopped sending?